In [1]:
import os
import sys
import gc
import math
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Optional, Literal
from numpy import linalg, ndarray

In [2]:
# Load MTP dataset
DATASET_PATH = os.path.expanduser('~/MTP/dataset.npy')
QUERY_PATH = os.path.expanduser('~/MTP/dataset_test.npy')
features = np.load(DATASET_PATH)
query_set = np.load(QUERY_PATH)
print(f'Loaded features shape: {features.shape}, query_set shape: {query_set.shape}')

Loaded features shape: (390000, 1000), query_set shape: (3000, 1000)


In [3]:
# Use same MLGT parameters as original notebook
num_tables = 500
hash_bits = 10
input_dim = features.shape[1]
hash_threshold = 0
match_threshold = 20
num_pools = 5000
pools_per_point = 3
points_per_pool = 600
num_queries = min(10, query_set.shape[0])
algorithm = 'my_algo'

In [6]:
# Import MLGT and utility functions from main notebook (assumes mlgt.py is in PYTHONPATH)
from sim_hash import SimHash
from mlgt import MLGT

In [7]:
def my_matching_algo(
    pooling_matrix: ndarray,
    positive_pools: ndarray
) -> ndarray:
    """
    My implementation of matching algorithm, to solve for sparse vector x given y = Ax, 
        where y is the positive_pools and A is the pooling_matrix.
    
    :param pooling_matrix: The pooling matrix A
    :type pooling_matrix: ndarray
    :param positive_pools: The output vector y
    :type positive_pools: ndarray
    :return: The recovered sparse vector x
    :rtype: ndarray
    """
    num_pools, num_features = pooling_matrix.shape
    assert positive_pools.shape == (num_pools,), "Incompatible shapes between pooling matrix and positive pools"
    possible = np.ones(num_features, dtype=bool)
    candidates = np.zeros(num_features, dtype=bool)
    print(f"[my_matching_algo] Initial possible.sum(): {possible.sum()} (should be {num_features})")
    # Eliminate features present in any negative pool
    for pool_idx in range(num_pools):
        if positive_pools[pool_idx] == 0:
            before = possible.sum()
            possible &= ~pooling_matrix[pool_idx]
            after = possible.sum()
            if before != after:
                print(f"  Eliminated {before - after} features from negative pool {pool_idx}")
    print(f"[my_matching_algo] After negative pool elimination, possible.sum(): {possible.sum()}")
    iter_count = 0
    while True:
        newly_identified = np.zeros(num_features, dtype=bool)
        for pool_idx in range(num_pools):
            if not positive_pools[pool_idx]:
                continue
            pool_points = pooling_matrix[pool_idx] & possible
            if np.sum(pool_points) == 1:
                newly_identified |= pool_points
        
        if not np.any(newly_identified):
            print(f"[my_matching_algo] Iter {iter_count}: No new identifications, breaking loop.")
            break
        print(f"[my_matching_algo] Iter {iter_count}: Identified {newly_identified.sum()} new candidates.")
        candidates |= newly_identified
        possible &= ~newly_identified
        identified_pools = pooling_matrix @ newly_identified
        positive_pools &= ~identified_pools
        print(f"[my_matching_algo] Iter {iter_count}: Remaining possible.sum(): {possible.sum()}, Remaining positive_pools.sum(): {positive_pools.sum()}")
        iter_count += 1
    print(f"[my_matching_algo] Final candidates.sum(): {candidates.sum()}")
    return candidates

In [9]:
mlgt = MLGT(
    num_tables=num_tables,
    hash_bits=hash_bits,
    input_dim=input_dim,
    hash_threshold=hash_threshold,
    match_threshold=match_threshold,
    num_pools=5000,
    pools_per_point=3,
    points_per_pool=234,
    features=features
)
mlgt.build_index()

Creating random pool assignment...
Assigned 0 points
Assigned 100000 points
Assigned 200000 points
Assigned 300000 points
Assigned 400000 points
Assigned 500000 points
Assigned 600000 points
Assigned 700000 points
Assigned 800000 points
Assigned 900000 points
Assigned 1000000 points
Assigned 1100000 points
DEBUG: Pools shape: (5000, 234), Pooling matrix shape: (5000, 390000)
DEBUG: Hash features shape: (390000, 500)
DEBUG: Computing inverted hash tables...
Building per-pool inverted index (CSR optimized)...
Built 0 pool indices
Built 500 pool indices
Built 1000 pool indices
Built 1500 pool indices
Built 2000 pool indices
Built 2500 pool indices
Built 3000 pool indices
Built 3500 pool indices
Built 4000 pool indices
Built 4500 pool indices


In [10]:
dataset_hashes: ndarray = mlgt.hash_features
query_hashes: ndarray = mlgt.hash_function.hash_bits_to_value(mlgt.hash_function(query_set)).astype(np.int32)
assert dataset_hashes.shape == (mlgt.num_features, mlgt.num_tables)
assert query_hashes.shape[1] == mlgt.num_tables
mean_precision: float = 0.0
mean_recall: float = 0.0

for qidx in tqdm(range(min(num_queries, query_set.shape[0])), desc="Validating queries..."):
    true_matches: ndarray = (dataset_hashes == query_hashes[qidx]).astype(np.int32).sum(axis=1) >= mlgt.match_threshold
    pooling_matrix: ndarray = mlgt.pooling_matrix
    positive_pools: ndarray = (pooling_matrix @ true_matches).astype(bool)
    # Solve using my_algo
    candidates = my_matching_algo(
        pooling_matrix=pooling_matrix,
        positive_pools=positive_pools
    )
    candidate_indices = np.where(candidates)[0]
    true_indices = np.where(true_matches)[0]
    intersection = len(set(candidate_indices).intersection(set(true_indices)))
    precision: float = intersection / len(candidate_indices) if len(candidate_indices) > 0 else 1.0
    recall: float = intersection / len(true_indices) if len(true_indices) > 0 else 1.0
    print(f"Query {qidx}: my_algo precision: {precision:.4f}, recall: {recall:.4f}")
    print(f"  #true_matches: {len(true_indices)}, #positive_pools: {np.sum(positive_pools)}, #candidates: {len(candidate_indices)}")
    print(f"  true_indices[:10]: {true_indices[:10]}")
    print(f"  candidate_indices[:10]: {candidate_indices[:10]}")
    mean_precision += precision
    mean_recall += recall

# Pooling matrix diagnostics
print(f"Pooling matrix shape: {pooling_matrix.shape}")
print(f"Per-point pool memberships (min, max, mean): {np.min(np.sum(pooling_matrix, axis=0))}, {np.max(np.sum(pooling_matrix, axis=0))}, {np.mean(np.sum(pooling_matrix, axis=0)):.2f}")
print(f"Per-pool sizes (min, max, mean): {np.min(np.sum(pooling_matrix, axis=1))}, {np.max(np.sum(pooling_matrix, axis=1))}, {np.mean(np.sum(pooling_matrix, axis=1)):.2f}")

mean_precision /= num_queries
mean_recall /= num_queries
print(f"Overall my_algo precision: {mean_precision:.4f}, recall: {mean_recall:.4f}")

Validating queries...:   0%|          | 0/10 [00:00<?, ?it/s]

[my_matching_algo] Initial possible.sum(): 390000 (should be 390000)
  Eliminated 234 features from negative pool 3
  Eliminated 234 features from negative pool 5
  Eliminated 234 features from negative pool 7
  Eliminated 234 features from negative pool 13
  Eliminated 232 features from negative pool 17
  Eliminated 234 features from negative pool 20
  Eliminated 233 features from negative pool 38
  Eliminated 232 features from negative pool 40
  Eliminated 233 features from negative pool 48
  Eliminated 233 features from negative pool 50
  Eliminated 234 features from negative pool 54
  Eliminated 234 features from negative pool 56
  Eliminated 234 features from negative pool 61
  Eliminated 233 features from negative pool 64
  Eliminated 230 features from negative pool 67
  Eliminated 232 features from negative pool 73
  Eliminated 232 features from negative pool 78
  Eliminated 230 features from negative pool 79
  Eliminated 234 features from negative pool 84
  Eliminated 232 featu

Validating queries...:  10%|█         | 1/10 [00:02<00:23,  2.56s/it]

[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
Query 0: my_algo precision: 1.0000, recall: 0.0000
  #true_matches: 2687, #positive_pools: 4002, #candidates: 0
  true_indices[:10]: [ 305  987 1068 1204 1242 1249 1498 1539 1545 1734]
  candidate_indices[:10]: []
[my_matching_algo] Initial possible.sum(): 390000 (should be 390000)
  Eliminated 234 features from negative pool 0
  Eliminated 234 features from negative pool 1
  Eliminated 234 features from negative pool 2
  Eliminated 234 features from negative pool 3
  Eliminated 233 features from negative pool 4
  Eliminated 234 features from negative pool 5
  Eliminated 234 features from negative pool 6
  Eliminated 234 features from negative pool 7
  Eliminated 234 features from negative pool 8
  Eliminated 234 features from negative pool 9
  Eliminated 233 features from negative pool 10
  Eliminated 234 features from negative pool 11
  Eliminated 233 features from negative 

Validating queries...:  20%|██        | 2/10 [00:06<00:25,  3.21s/it]

...

[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
Query 8: my_algo precision: 1.0000, recall: 0.0000
  #true_matches: 1949, #positive_pools: 3447, #candidates: 0
  true_indices[:10]: [116 140 187 210 399 521 622 807 868 914]
  candidate_indices[:10]: []
[my_matching_algo] Initial possible.sum(): 390000 (should be 390000)
  Eliminated 234 features from negative pool 0
  Eliminated 234 features from negative pool 1
  Eliminated 234 features from negative pool 3
  Eliminated 233 features from negative pool 4
  Eliminated 234 features from negative pool 5
  Eliminated 234 features from negative pool 6
  Eliminated 234 features from negative pool 7
  Eliminated 234 features from negative pool 8
  Eliminated 234 features from negative pool 9
  Eliminated 233 features from negative pool 10
  Eliminated 234 features from negative pool 11
  Eliminated 233 features from negative pool 12
  Eliminated 234 features from negative pool 13
 

Validating queries...: 100%|██████████| 10/10 [00:34<00:00,  3.46s/it]

[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
Query 9: my_algo precision: 1.0000, recall: 0.0000
  #true_matches: 300, #positive_pools: 817, #candidates: 0
  true_indices[:10]: [ 395  460  698 3219 3429 3534 5134 5611 6378 7047]
  candidate_indices[:10]: []
Pooling matrix shape: (5000, 390000)


Per-point pool memberships (min, max, mean): 3, 3, 3.00


KeyboardInterrupt: 

## Run MLGT Algorithm Quality Test Suite (MTP)

In [ ]:
def test_mlgt_algorithms(mlgt, query_set, num_queries=10, top_k=100):
    results = {algo: [] for algo in ['my_algo', 'lsmr', 'lsqr']}
    for algo in ['my_algo', 'lsmr', 'lsqr']:
        print(f'Testing {algo}...')
        for qidx in range(min(num_queries, query_set.shape[0])):
            query_vector = query_set[qidx]
            precision, recall, recall_saff_hash, recall_hash_true, t_saff, t_hash, t_true = mlgt.get_metrics(query_vector, algorithm=algo)
            results[algo].append({
                'precision': precision,
                'recall': recall,
                'recall_vs_hash': recall_saff_hash,
                'recall_hash_vs_true': recall_hash_true,
                'f1': 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0,
                'time': t_saff
            })
            print(f'Query {qidx}: {algo} time {t_saff:.4f}s | Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {results[algo][-1][f1]:.4f}')
    return results

In [ ]:
def summarize_results(results):
    import pandas as pd
    summary = {}
    for algo, res_list in results.items():
        df = pd.DataFrame(res_list)
        summary[algo] = {
            'precision_mean': df['precision'].mean(),
            'recall_mean': df['recall'].mean(),
            'f1_mean': df['f1'].mean(),
            'time_mean': df['time'].mean(),
            'recall_vs_hash_mean': df['recall_vs_hash'].mean(),
            'recall_hash_vs_true_mean': df['recall_hash_vs_true'].mean()
        }
    return pd.DataFrame(summary).T

In [ ]:
results = test_mlgt_algorithms(mlgt, query_set, num_queries=num_queries, top_k=100)
summarize_results(results)

### Unit Test: Recovery on Small Random y = Ax Problems (MTP)

In [ ]:
def random_group_testing_problem(num_features=50, num_pools=30, sparsity=3, pools_per_point=3, points_per_pool=5, seed=42):
    np.random.seed(seed)
    pooling_matrix = np.zeros((num_pools, num_features), dtype=bool)
    for i in range(num_features):
        chosen = np.random.choice(num_pools, size=pools_per_point, replace=False)
        pooling_matrix[chosen, i] = 1
    for j in range(num_pools):
        if pooling_matrix[j].sum() < points_per_pool:
            missing = points_per_pool - pooling_matrix[j].sum()
            idxs = np.random.choice(np.where(~pooling_matrix[j])[0], size=missing, replace=False)
            pooling_matrix[j, idxs] = 1
    x_true = np.zeros(num_features, dtype=bool)
    nonzero = np.random.choice(num_features, size=sparsity, replace=False)
    x_true[nonzero] = 1
    y = (pooling_matrix @ x_true) > 0
    return pooling_matrix, y, x_true

In [ ]:
def test_group_testing_algos():
    pooling_matrix, y, x_true = random_group_testing_problem()
    print('True support:', np.where(x_true)[0])
    from copy import deepcopy
    y_my = deepcopy(y)
    mask_my = my_matching_algo(pooling_matrix.copy(), y_my.copy())
    recovered_my = np.where(mask_my)[0]
    print('my_algo recovered:', recovered_my)
    print('my_algo recall:', np.sum(mask_my & x_true) / np.sum(x_true), 'precision:', np.sum(mask_my & x_true) / max(np.sum(mask_my),1))
    from scipy.sparse import csr_matrix
    y_lsmr = deepcopy(y)
    A_sparse = csr_matrix(pooling_matrix)
    from scipy.sparse.linalg import lsmr
    sol = lsmr(A_sparse, y_lsmr.astype(float))
    x_lsmr = sol[0]
    recovered_lsmr = np.where(x_lsmr > 0.5)[0]
    print('lsmr recovered:', recovered_lsmr)
    print('lsmr recall:', np.sum(x_true[recovered_lsmr]) / np.sum(x_true), 'precision:', np.sum(x_true[recovered_lsmr]) / max(len(recovered_lsmr),1))
    from scipy.sparse.linalg import lsqr
    sol2 = lsqr(A_sparse, y.astype(float))
    x_lsqr = sol2[0]
    recovered_lsqr = np.where(x_lsqr > 0.5)[0]
    print('lsqr recovered:', recovered_lsqr)
    print('lsqr recall:', np.sum(x_true[recovered_lsqr]) / np.sum(x_true), 'precision:', np.sum(x_true[recovered_lsqr]) / max(len(recovered_lsqr),1))

In [ ]:
test_group_testing_algos()